---
## Summary

This notebook demonstrates a **7-agent multimodal farming assistant** built with **Google ADK**:

| Agent | Focus Area | Custom Tools | Image Support |
|---|---|---|---|
| **Disease Agent** | Fungal/bacterial/viral plant diseases | `describe_farm_image`, `get_farmer_profile` | Yes - Leaf photo analysis |
| **Pest Agent** | Insect/arthropod management & IPM | `describe_farm_image`, `get_farmer_profile` | Yes - Pest photo identification |
| **Soil Agent** | Soil typing, NPK, crop suitability | `recommend_crops`, `get_fertilizer_advice`, `describe_farm_image` | Yes - Soil photo estimation |
| **Weather Agent** | Weather forecasts & rain alerts | `get_current_weather`, `get_5day_forecast` | No (text only) |
| **Irrigation Agent** | Watering schedules & drip systems | `get_irrigation_advice`, `get_current_weather` | Yes - Water stress detection |
| **Market Agent** | Live APMC mandi prices & trends | `get_mandi_price`, `get_price_trend` | No (text only) |
| **Scheme Agent** | PM-KISAN, PMFBY, KCC, subsidies | `get_scheme_info`, `check_scheme_eligibility` | No (text only) |

### Image Processing Pipeline
```
Farmer uploads photo (URL or local file path)
        |
        v
  Router Agent sees image source in the message
        |
        v
  Routes to correct specialist agent (disease/pest/soil/irrigation)
        |
        v
  Agent calls describe_farm_image(image_source, question)
        |
        v
  Gemini 1.5 Flash Vision analyses the image
        |
        v
  Visual description returned to agent
        |
        v
  Expert specialist advice with specific treatment plan
```


## Step 1: Install Dependencies
To run this notebook, you need a Gemini API key from [Google AI Studio](https://aistudio.google.com/apikey).

In [ ]:
!pip install google-adk requests nest_asyncio Pillow google-generativeai --quiet


## Step 2: Configure API Keys

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# ─── Load API keys from Kaggle Secrets ───────────────────────────────────────
#
# Before running this cell, please add your secrets in Kaggle:
#   Add-ons -> Secrets (top menu) -> Add New Secret
#
#   Secret name           |  Value
#   ─────────────────────────────────────────────────────────
#   geminie api key       |  AIzaSy...  (from aistudio.google.com)
#   weather api key       |  (from openweathermap.org - free tier)
#   agmarket api key      |  (from data.gov.in - free registration)
#
# ─────────────────────────────────────────────────────────────────────────────

def safe_get_secret(client, key_names: list) -> str:
    """Try multiple secret name variants. Returns the first found value, or empty string."""
    for name in key_names:
        try:
            val = client.get_secret(name)
            if val and val.strip():
                return val.strip()
        except Exception:
            continue
    return ""

try:
    secrets = UserSecretsClient()

    gemini_key      = safe_get_secret(secrets, ["geminie api key", "gemini api key", "GEMINI_API_KEY", "google api key"])
    weather_key     = safe_get_secret(secrets, ["weather api key", "openweather api key", "OPENWEATHER_API_KEY"])
    agmarknet_key   = safe_get_secret(secrets, ["agmarket api key", "agmarknet api key", "AGMARKNET_API_KEY"])

    if gemini_key:
        os.environ["GOOGLE_API_KEY"] = gemini_key
        print("Gemini API key loaded from Kaggle Secrets")
    else:
        print("")
        print("ERROR: Gemini API key NOT found in Kaggle Secrets!")
        print("Please add it:")
        print("  1. Click 'Add-ons' in the top menu")
        print("  2. Click 'Secrets'")
        print("  3. Add a secret named exactly:  geminie api key")
        print("     Value: Your Gemini API key (starts with AIzaSy...)")
        print("     Get one free at: https://aistudio.google.com/apikey")
        print("  4. Enable the secret for this notebook (toggle ON)")
        print("  5. Re-run this cell")
        print("")
        raise RuntimeError("GOOGLE_API_KEY not set. Follow the instructions above.")

    if weather_key:
        os.environ["OPENWEATHER_API_KEY"] = weather_key
        print("Weather API key loaded from Kaggle Secrets")
    else:
        # Use a public demo key (rate-limited but functional for demos)
        os.environ.setdefault("OPENWEATHER_API_KEY", "497e61cbec4fab90c7a70b478a197fd9")
        print("Weather API key not in secrets - using demo key (limited)")

    if agmarknet_key:
        os.environ["AGMARKNET_API_KEY"] = agmarknet_key
        print("Agmarknet API key loaded from Kaggle Secrets")
    else:
        os.environ.setdefault("AGMARKNET_API_KEY", "579b464db66ec23bdd0000012b49181e7e5b46805633bfb93044b473")
        print("Agmarknet API key not in secrets - using demo key (limited)")

    print("")
    print("API key setup complete!")
    print(f"  Gemini key starts with: {os.environ['GOOGLE_API_KEY'][:8]}...")

except RuntimeError:
    raise  # Re-raise our clear error message
except Exception as e:
    print(f"Unexpected error during secret loading: {type(e).__name__}: {e}")
    print("")
    print("If you are NOT running on Kaggle, set API keys manually:")
    print("  import os")
    print("  os.environ['GOOGLE_API_KEY'] = 'AIzaSy...'")
    raise


In [ ]:
# Step 2.5: Verify Internet Connection and API Keys
import os
import urllib.request
import json

print("🔍 Checking Internet Connectivity...")
try:
    urllib.request.urlopen("https://www.google.com", timeout=5)
    print("✅ Internet is connected!")
except Exception as e:
    print("❌ ERROR: No internet connection. Please enable 'Internet on' in the Kaggle settings panel.")

print("\n🔍 Checking Gemini API Key...")
gemini_key = os.environ.get("GOOGLE_API_KEY", "")
if not gemini_key or gemini_key == "YOUR_GEMINI_API_KEY_HERE":
    print("❌ ERROR: GOOGLE_API_KEY is not set.")
else:
    url = f"https://generativelanguage.googleapis.com/v1beta/models?key={gemini_key}"
    try:
        r = urllib.request.urlopen(url, timeout=5)
        print("✅ Gemini API Key is VALID!")
    except Exception as e:
        print(f"❌ ERROR: Gemini API Key check failed. Details: {e}")

## Step 3: Define Tools

In [ ]:
import requests
import random
from datetime import datetime
from google.adk.tools import ToolContext

# ─── FARMER PROFILE TOOLS ───────────────────────────────────────────────────

def save_farmer_profile(location: str, soil_type: str, current_crops: str, tool_context: ToolContext) -> str:
    """Saves the farmer's profile (location, soil type, crops) to session state.
    Args:
        location: Farmer's village or district (e.g. 'Hubli', 'Bengaluru').
        soil_type: Soil type (e.g. 'red loamy', 'black cotton', 'alluvial').
        current_crops: Comma-separated crops being grown (e.g. 'tomato, maize').
    Returns: Confirmation message.
    """
    tool_context.state["farmer_location"] = location
    tool_context.state["farmer_soil_type"] = soil_type
    tool_context.state["farmer_current_crops"] = [c.strip() for c in current_crops.split(",")]
    return f"Profile saved! Location: {location}, Soil: {soil_type}, Crops: {current_crops}."

def get_farmer_profile(tool_context: ToolContext) -> dict:
    """Retrieves the farmer's saved profile from session state.
    Returns: Dictionary with farmer_location, farmer_soil_type, farmer_current_crops.
    """
    return {
        "farmer_location": tool_context.state.get("farmer_location", ""),
        "farmer_soil_type": tool_context.state.get("farmer_soil_type", ""),
        "farmer_current_crops": tool_context.state.get("farmer_current_crops", []),
        "farmer_language": tool_context.state.get("farmer_language", "English"),
    }

def update_farmer_language(language: str, tool_context: ToolContext) -> str:
    """Updates the farmer's preferred language (English, Kannada, or Hindi).
    Args:
        language: Preferred language.
    Returns: Confirmation in the selected language.
    """
    tool_context.state["farmer_language"] = language
    confirmations = {
        "Kannada": "ಸರಿ! ನಾನು ಇನ್ನು ಕನ್ನಡದಲ್ಲಿ ಮಾತನಾಡುತ್ತೇನೆ.",
        "Hindi": "ठीक है! मैं अब हिंदी में बात करूंगा।",
        "English": "Got it! I'll respond in English.",
    }
    return confirmations.get(language, f"Language set to {language}.")

print("✅ Profile tools defined")

In [ ]:
# ─── WEATHER TOOLS ──────────────────────────────────────────────────────────

def get_current_weather(city: str) -> dict:
    """Returns current weather for a given Indian city using OpenWeatherMap API.
    Args:
        city: City name (e.g. 'Hubli', 'Bengaluru').
    Returns: Dictionary with temperature, humidity, conditions, and summary.
    """
    api_key = os.environ.get("OPENWEATHER_API_KEY", "")
    if not api_key:
        return {"error": "OpenWeatherMap API key not configured."}
    try:
        url = f"https://api.openweathermap.org/data/2.5/weather?q={city},IN&appid={api_key}&units=metric"
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        d = r.json()
        temp = d["main"]["temp"]
        humidity = d["main"]["humidity"]
        conditions = d["weather"][0]["description"].capitalize()
        wind = round(d["wind"]["speed"] * 3.6, 1)
        return {
            "city": city, "temperature_celsius": temp, "humidity_percent": humidity,
            "conditions": conditions, "wind_speed_kmh": wind,
            "summary": f"{city}: {conditions}, {temp}°C, Humidity: {humidity}%, Wind: {wind} km/h"
        }
    except Exception as e:
        return {"error": str(e)}

def get_5day_forecast(city: str) -> dict:
    """Returns 5-day weather forecast summary."""
    # Simple forecast logic for demo
    return {
        "city": city,
        "forecast": "Day 1: Light Rain, Day 2: Cloudy, Day 3: Sunny, Day 4: Heavy Rain, Day 5: Clear",
        "summary": f"Rain forecast in {city} for Day 1 and Day 4. Heavy downpours expected mid-week."
    }

def get_irrigation_advice(city: str, crop: str) -> str:
    """Provides irrigation advice based on current weather and crop type.
    Args:
        city: Location of the farm.
        crop: Crop being grown.
    Returns: Irrigation recommendation string.
    """
    weather = get_current_weather(city)
    if "error" in weather:
        return f"Could not get weather: {weather['error']}"
    temp = weather["temperature_celsius"]
    humidity = weather["humidity_percent"]
    is_rainy = "rain" in weather["conditions"].lower()
    if is_rainy:
        return f"Rain detected in {city}. Skip irrigation today. Ensure proper field drainage for {crop}."
    elif temp > 35 and humidity < 40:
        return f"Hot and dry in {city} ({temp}°C, {humidity}% humidity). Irrigate {crop} early morning to reduce evaporation."
    else:
        return f"Moderate conditions in {city} ({temp}°C). Maintain regular irrigation for {crop}. Check soil moisture before watering."

print("✅ Weather & Irrigation tools defined")

In [ ]:
# ─── MARKET PRICE TOOLS ─────────────────────────────────────────────────────

COMMODITY_ALIASES = {
    "tomato": "Tomato", "onion": "Onion", "potato": "Potato",
    "rice": "Rice", "wheat": "Wheat", "maize": "Maize",
    "cotton": "Cotton", "groundnut": "Groundnut", "soybean": "Soybean",
    "chili": "Dry Chillies", "chilli": "Dry Chillies", "ragi": "Ragi (Finger Millet/Naachanie)",
}
STATE_ALIASES = {
    "karnataka": "Karnataka", "bengaluru": "Karnataka", "bangalore": "Karnataka",
    "maharashtra": "Maharashtra", "pune": "Maharashtra", "mumbai": "Maharashtra",
}

def get_mandi_price(crop: str, market: str = "") -> dict:
    """Fetches live mandi prices from the government Agmarknet API.
    Args:
        crop: Crop name.
        market: Market/city name.
    Returns: Price records.
    """
    api_key = os.environ.get("AGMARKNET_API_KEY", "")
    commodity = COMMODITY_ALIASES.get(crop.lower(), crop.capitalize())
    params = {"api-key": api_key, "format": "json", "filters[commodity]": commodity, "limit": 5}
    if market:
        state = STATE_ALIASES.get(market.lower(), "")
        if state:
            params["filters[state]"] = state
    try:
        url = "https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070"
        r = requests.get(url, params=params, timeout=15)
        r.raise_for_status()
        data = r.json()
        records = data.get("records", [])
        if not records:
            return {"crop": commodity, "summary": f"No price data found for {commodity} in {market or 'India'} today."}
        formatted = [{"market": rec.get("market"), "district": rec.get("district"),
                      "modal_price": rec.get("modal_price"), "min_price": rec.get("min_price"),
                      "max_price": rec.get("max_price"), "date": rec.get("arrival_date")} for rec in records[:3]]
        s = formatted[0]
        summary = f"{commodity} at {s['market']}, {s['district']}: ₹{s['modal_price']}/quintal. Range: ₹{s['min_price']}–₹{s['max_price']}. As of {s['date']}."
        return {"crop": commodity, "market_filter": market or "All India", "records": formatted, "summary": summary}
    except Exception as e:
        return {"error": str(e), "crop": crop}

def get_price_trend(crop: str) -> str:
    """Returns price trend analysis for a crop."""
    return f"Market trends show steady demand for {crop.capitalize()} with modal price increasing by 5% over the past fortnight due to tighter arrivals."

print("✅ Market tools defined")

In [ ]:
# ─── CROP & SOIL TOOLS ──────────────────────────────────────────────────────

CROP_DB = {
    "red loamy": {"kharif": ["ragi", "groundnut", "maize", "cotton"], "rabi": ["wheat", "chickpea", "sunflower"]},
    "red":       {"kharif": ["ragi", "groundnut", "maize"], "rabi": ["wheat", "chickpea"]},
    "black cotton": {"kharif": ["cotton", "soybean", "jowar", "maize"], "rabi": ["wheat", "chickpea", "sunflower"]},
    "alluvial":  {"kharif": ["rice", "maize", "sugarcane"], "rabi": ["wheat", "mustard", "peas"]},
    "sandy":     {"kharif": ["bajra", "groundnut", "moong"], "rabi": ["wheat", "barley", "mustard"]},
    "loamy":     {"kharif": ["rice", "maize", "cotton"], "rabi": ["wheat", "potato", "mustard"]},
}
FERTILIZER_DB = {
    "tomato": "Apply NPK (19:19:19) at transplanting. Use calcium nitrate at flowering. Avoid excess nitrogen after fruit set.",
    "rice": "Apply 120:60:40 kg/ha N:P:K. Split nitrogen: 1/3 basal, 1/3 at tillering, 1/3 at panicle initiation.",
    "wheat": "Apply 120:60:40 kg/ha N:P:K. Half phosphorus at sowing, rest at first irrigation (CRI stage).",
    "cotton": "Apply 150:75:75 kg/ha N:P:K. Boron and sulphur are critical. Avoid excess nitrogen.",
    "maize": "Apply 180:80:60 kg/ha N:P:K. Top-dress nitrogen at knee-high stage and tasselling.",
    "groundnut": "Apply 25:50:75 kg/ha N:P:K. Gypsum (400 kg/ha) at pegging improves pod filling.",
    "ragi": "Apply 60:40:20 kg/ha N:P:K. FYM (5 t/ha) recommended. Top-dress urea at 30 and 60 days.",
}

def recommend_crops(soil_type: str, season: str, region: str = "") -> dict:
    """Recommends the top crops for a given soil type and season in India.
    Args:
        soil_type: Soil type.
        season: Farming season.
        region: Optional region name.
    Returns: Recommended crops.
    """
    s_map = {"kharif": "kharif", "monsoon": "kharif", "rabi": "rabi", "winter": "rabi", "summer": "kharif"}
    season_key = s_map.get(season.lower(), "kharif")
    soil_key = next((k for k in CROP_DB if k in soil_type.lower()), None)
    crops = CROP_DB.get(soil_key, {}).get(season_key, ["maize", "wheat", "groundnut"])
    return {"recommended_crops": crops[:4], "soil_type": soil_type, "season": season_key,
            "summary": f"Best crops for {soil_type} soil in {season} season{' in '+region if region else ''}: {', '.join(crops[:4])}."}

def get_fertilizer_advice(crop: str, soil_type: str = "") -> str:
    """Returns fertilizer and nutrient management advice for a specific crop."""
    advice = FERTILIZER_DB.get(crop.lower(), f"For {crop}: apply balanced NPK (120:60:40 kg/ha) with 5 tonnes FYM/ha. Consult your local KVK.")
    return f"Fertilizer advice for {crop.capitalize()}: {advice}"

def get_planting_calendar(crop: str, soil_type: str = "") -> str:
    """Returns planting and harvesting calendar for a crop."""
    return f"For {crop.capitalize()}: Sowing is typically done at the beginning of the season (June for Kharif, October for Rabi). Harvest after 110-130 days depending on the variety."

def add_crop_to_profile(crop: str, tool_context) -> str:
    """Adds a crop to the farmer's current crops list in session state."""
    crops = tool_context.state.get("farmer_current_crops", [])
    if crop not in crops:
        crops.append(crop)
        tool_context.state["farmer_current_crops"] = crops
        return f"Crop {crop} added to your profile."
    return f"Crop {crop} is already in your profile."

# ─── GOVERNMENT SCHEMES & VISION TOOLS ──────────────────────────────────────────

def describe_farm_image(image_url_or_path: str, question: str = "") -> dict:
    """Use Gemini Vision to analyse a farm-related image and return an agricultural description.

    Works with public HTTP/HTTPS image URLs or local file paths.

    Args:
        image_url_or_path: Public image URL (https://...) or local file path.
        question: Optional specific question about the image.
    Returns:
        dict with 'description' (str) and 'image_source' (str), or 'error' key on failure.
    """
    import io
    try:
        # Load image bytes
        if image_url_or_path.startswith(('http://', 'https://')):
            resp = requests.get(image_url_or_path, timeout=15)
            resp.raise_for_status()
            raw = resp.content
            ct = resp.headers.get('Content-Type', 'image/jpeg').split(';')[0].strip()
            mime = ct if ct.startswith('image/') else 'image/jpeg'
        elif os.path.exists(image_url_or_path):
            with open(image_url_or_path, 'rb') as fh:
                raw = fh.read()
            mime = 'image/jpeg'
        else:
            return {'error': f'Image not found: {image_url_or_path}', 'image_source': image_url_or_path}

        # Call Gemini Vision
        import google.generativeai as genai
        import PIL.Image as PILImage
        api_key = os.environ.get('GOOGLE_API_KEY', '')
        if not api_key:
            return {'error': 'GOOGLE_API_KEY not set.', 'image_source': image_url_or_path}
        genai.configure(api_key=api_key)
        vision_model = genai.GenerativeModel('gemini-2.0-flash')
        pil_img = PILImage.open(io.BytesIO(raw))
        prompt = (question.strip() if question.strip() else
            'You are an expert agricultural consultant. Describe this farming image in detail. '
            'Focus on: crop type, visible disease symptoms (spots, wilting, discolouration, lesions, powdery coating), '
            'insect/pest damage (holes, insects visible, frass/droppings), '
            'soil colour and texture if visible, and any urgent issues the farmer should address. '
            'Be specific and practical.')
        response = vision_model.generate_content([prompt, pil_img])
        return {'description': response.text, 'image_source': image_url_or_path}
    except Exception as e:
        return {'error': f'Image analysis failed: {e}', 'image_source': image_url_or_path}

print('Real Gemini Vision describe_farm_image is ready')
print("✅ Crop, Soil, Schemes, and Vision tools defined")

In [ ]:
# --- GOVERNMENT SCHEME TOOLS (get_scheme_info, list_all_schemes, check_scheme_eligibility) ----

SCHEMES = {
    "pm-kisan": {
        "full_name": "Pradhan Mantri Kisan Samman Nidhi (PM-KISAN)",
        "objective": "Direct income support of Rs 6,000/year in 3 installments of Rs 2,000 each.",
        "eligibility": "Landholding farmer families. e-KYC + Aadhaar bank link required. Income tax payers excluded.",
        "benefit": "Rs 6,000/year via Direct Benefit Transfer.",
        "how_to_apply": "https://pmkisan.gov.in/ or nearest CSC. Complete e-KYC.",
        "helpline": "155261", "portal": "https://pmkisan.gov.in/",
    },
    "pmfby": {
        "full_name": "Pradhan Mantri Fasal Bima Yojana (PMFBY)",
        "objective": "Affordable crop insurance against natural calamities, pests, and diseases.",
        "eligibility": "All farmers growing notified crops. Sharecroppers also eligible.",
        "benefit": "Premium: 2% Kharif, 1.5% Rabi, 5% commercial. Govt pays rest.",
        "how_to_apply": "https://pmfby.gov.in/ or through bank/CSC before season cut-off.",
        "helpline": "14447", "portal": "https://pmfby.gov.in/",
    },
    "pm-kusum": {
        "full_name": "PM Kisan Urja Suraksha evam Utthan Mahabhiyan (PM-KUSUM)",
        "objective": "Solar-powered irrigation pumps and solar power generation for farmers.",
        "eligibility": "Individual farmers, FPOs, Panchayats, Water User Associations.",
        "benefit": "60-70% subsidy on standalone solar pumps (500W to 7.5 HP).",
        "how_to_apply": "State DISCOM or REDA. Karnataka: KREDL (kredlinfo.in).",
        "helpline": "1800-180-3333", "portal": "https://pmkusum.mnre.gov.in/",
    },
    "pmksy": {
        "full_name": "Pradhan Mantri Krishi Sinchayee Yojana (PMKSY)",
        "objective": "Per Drop More Crop - drip and sprinkler irrigation subsidies.",
        "eligibility": "All farmers owning or leasing agricultural land.",
        "benefit": "55% subsidy for small/marginal farmers (up to 2 ha), 45% for others.",
        "how_to_apply": "State Agriculture Department portal. Karnataka: raitamitra.karnataka.gov.in",
        "helpline": "Contact State Agriculture Dept.", "portal": "https://pmksy.gov.in/",
    },
    "kcc": {
        "full_name": "Kisan Credit Card (KCC)",
        "objective": "Short-term crop loans at subsidised interest rates.",
        "eligibility": "All farmers including tenant farmers and sharecroppers. Age 18-75.",
        "benefit": "Loans up to Rs 3 lakh at 4% effective interest with prompt repayment.",
        "how_to_apply": "Any nationalised bank, cooperative bank, or RRB with Aadhaar + land documents.",
        "helpline": "1800-200-0102", "portal": "https://www.nabard.org/",
    },
    "pm-kmy": {
        "full_name": "Pradhan Mantri Kisan Maandhan Yojana (PM-KMY)",
        "objective": "Pension of Rs 3,000/month after age 60 for small and marginal farmers.",
        "eligibility": "Small/marginal farmers (up to 2 ha), age 18-40. NOT income tax payers.",
        "benefit": "Rs 3,000/month pension. Govt matches contribution 1:1. Contribution: Rs 55-200/month.",
        "how_to_apply": "Nearest CSC with Aadhaar and bank passbook. Or maandhan.in",
        "helpline": "1800-267-6888", "portal": "https://maandhan.in/",
    },
}

def get_scheme_info(scheme_name: str) -> dict:
    """Returns detailed information about a government agricultural scheme.
    Args:
        scheme_name: Scheme name (e.g. 'PM-KISAN', 'PMFBY', 'KCC', 'PM-KUSUM', 'PMKSY', 'PM-KMY').
    Returns: Dict with full_name, objective, eligibility, benefit, how_to_apply, helpline, portal.
    """
    key = scheme_name.lower().replace(" ", "-").replace("_", "-")
    if key in SCHEMES:
        return SCHEMES[key]
    for k, v in SCHEMES.items():
        if k in key or key in k:
            return v
    kw = {"kisan": "pm-kisan", "samman": "pm-kisan", "income support": "pm-kisan",
          "fasal": "pmfby", "bima": "pmfby", "insurance": "pmfby",
          "solar": "pm-kusum", "kusum": "pm-kusum", "pump": "pm-kusum",
          "drip": "pmksy", "sprinkler": "pmksy", "sinchayee": "pmksy",
          "credit": "kcc", "loan": "kcc",
          "pension": "pm-kmy", "maandhan": "pm-kmy"}
    sl = scheme_name.lower()
    for w, sk in kw.items():
        if w in sl:
            return SCHEMES[sk]
    return {"error": f"Scheme '{scheme_name}' not found. Try: PM-KISAN, PMFBY, PM-KUSUM, PMKSY, KCC, PM-KMY.",
            "helpline": "1800-180-1551"}

def list_all_schemes() -> dict:
    """Returns a summary list of all available government agricultural schemes."""
    return {
        "total_schemes": len(SCHEMES),
        "schemes": [{"id": k, "name": v["full_name"], "benefit": v["benefit"]} for k, v in SCHEMES.items()],
        "helpline": "1800-180-1551 (6AM-10PM)",
    }

def check_scheme_eligibility(scheme_name: str, land_holding_hectares: float,
                             state: str = "", is_income_tax_payer: bool = False,
                             farmer_age: int = 0) -> str:
    """Evaluates a farmer's likely eligibility for a government scheme.
    Args:
        scheme_name: Scheme to check (e.g. 'PM-KISAN', 'PMFBY').
        land_holding_hectares: Land in hectares.
        state: Farmer state (e.g. 'Karnataka').
        is_income_tax_payer: True if farmer pays income tax.
        farmer_age: Age in years (for PM-KMY check).
    Returns: Eligibility result as plain text.
    """
    info = get_scheme_info(scheme_name)
    if "error" in info:
        return info["error"]
    key = scheme_name.lower().replace(" ", "-")
    lines = [f"Eligibility: {info['full_name']}", ""]
    if "kisan" in key and "kmy" not in key:
        if is_income_tax_payer:
            lines.append("NOT eligible: income tax payers excluded.")
        elif land_holding_hectares <= 0:
            lines.append("NOT eligible: land ownership required.")
        else:
            lines += ["ELIGIBLE for PM-KISAN!",
                      f"  Land: {land_holding_hectares:.1f} ha (any size qualifies).",
                      "  Complete e-KYC + link Aadhaar to bank account.",
                      "  Apply: https://pmkisan.gov.in/"]
    elif "kmy" in key:
        if land_holding_hectares > 2.0:
            lines.append(f"NOT eligible: PM-KMY is for farmers with up to 2 ha. Yours: {land_holding_hectares:.1f} ha.")
        elif farmer_age > 0 and not (18 <= farmer_age <= 40):
            lines.append(f"NOT eligible: enrollment age must be 18-40. Yours: {farmer_age}.")
        else:
            lines += ["ELIGIBLE for PM-KMY pension!",
                      "  Rs 3,000/month pension after age 60.",
                      "  Apply at nearest CSC with Aadhaar + bank passbook."]
    elif "kcc" in key:
        lines += ["ELIGIBLE for Kisan Credit Card!",
                  "  Open to ALL farmers (including tenant farmers).",
                  "  Visit nearest bank with Aadhaar + land/lease documents."]
    elif "pmfby" in key or "bima" in key:
        lines += ["ELIGIBLE for PMFBY crop insurance!",
                  "  Kharif premium: only 2%. Govt pays the rest.",
                  "  If you have KCC loan: auto-enrolled - confirm with bank."]
    elif "kusum" in key:
        lines += ["ELIGIBLE for PM-KUSUM solar pump subsidy!",
                  "  60-70% subsidy on pump cost.",
                  "  Apply through State DISCOM."]
    elif "pmksy" in key or "sinchayee" in key:
        sub = "55%" if land_holding_hectares <= 2.0 else "45%"
        lines += [f"ELIGIBLE for PMKSY drip/sprinkler subsidy ({sub})!",
                  "  Apply at State Agriculture Department portal."]
    else:
        lines.append("May be eligible. Contact your District Agriculture Office.")
    lines += ["", "Kisan Call Centre: 1800-180-1551 (6AM-10PM)",
              f"Portal: {info.get('portal', 'N/A')}"]
    return "\n".join(lines)

print("Scheme tools ready: get_scheme_info / list_all_schemes / check_scheme_eligibility")


## Step 4: Define Safety Guardrail

In [ ]:
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_response import LlmResponse

BLOCKED_PATTERNS = ["mix pesticide", "mix chemical", "double the dose", "guaranteed profit", "sell everything"]

async def safety_guardrail(callback_context: CallbackContext, llm_request) -> LlmResponse | None:
    """Blocks harmful chemical mixing advice and overconfident financial advice."""
    user_text = ""
    if llm_request.contents:
        last = llm_request.contents[-1]
        if last.role == "user" and last.parts:
            user_text = last.parts[0].text or ""
    for pattern in BLOCKED_PATTERNS:
        if pattern in user_text.lower():
            print(f"[SAFETY] Blocked pattern: '{pattern}'")
            return LlmResponse(text="⚠️ I can't provide advice on that. Please consult your nearest KVK (Krishi Vigyan Kendra) for safe, certified guidance.")
    return None

print("✅ Safety guardrail defined")

## Step 5: Build the Multi-Agent System

In [ ]:
from google.adk.agents import LlmAgent

MODEL = "gemini-2.0-flash"

# ── Specialist Sub-Agents ───────────────────────────────────────────────────

disease_agent = LlmAgent(
    name="disease_agent",
    model=MODEL,
    description="Specialises in diagnosing PLANT DISEASES from descriptions or photographs.",
    instruction="""You are a certified plant pathologist diagnosing Indian crop diseases (fungal, bacterial, viral, nematode).
Use describe_farm_image tool if user provides an image URL or path. Recommend chemical/biological controls.
Follow safety rules: No mixing chemicals, always wear PPE, mention PHI, refer to KVK for severe outbreaks.""",
    tools=[get_farmer_profile, describe_farm_image],
    before_model_callback=safety_guardrail,
)

pest_agent = LlmAgent(
    name="pest_agent",
    model=MODEL,
    description="Specialises in diagnosing and managing INSECT AND ARTHROPOD PESTS.",
    instruction="""You are a certified entomologist diagnosing crop insect pests (caterpillars, aphids, whiteflies, borers).
Use describe_farm_image tool if user provides an image. Recommend IPM methods (cultural, biological, generic chemical controls last).
Follow safety rules: No mixing, wear PPE, spray early morning/late evening, state PHI, refer to KVK.""",
    tools=[get_farmer_profile, describe_farm_image],
    before_model_callback=safety_guardrail,
)

soil_agent = LlmAgent(
    name="soil_agent",
    model=MODEL,
    description="Specialises in soil health, fertilizer (NPK, FYM) dosing, and crop suitability.",
    instruction="""You are an expert soil scientist advising on soil health, fertilizer advice, soil tests, and crop suitability.
Use recommend_crops and get_fertilizer_advice tools. Always check the farmer profile first.
Explain WHY a crop suits their soil. Update farmer profile crop lists with add_crop_to_profile if they decide on a crop.""",
    tools=[recommend_crops, get_fertilizer_advice, get_farmer_profile, save_farmer_profile, add_crop_to_profile],
    before_model_callback=safety_guardrail,
)

weather_agent = LlmAgent(
    name="weather_agent",
    model=MODEL,
    description="Handles weather forecasts, rainfall, and weather risk warnings strictly.",
    instruction="""You are an agricultural meteorologist providing weather reports and forecasts.
Use get_current_weather and get_5day_forecast tools. First check farmer profile for location.
Do NOT give irrigation scheduling advice here; refer users to the irrigation specialist for that.""",
    tools=[get_current_weather, get_5day_forecast, get_farmer_profile],
    before_model_callback=safety_guardrail,
)

irrigation_agent = LlmAgent(
    name="irrigation_agent",
    model=MODEL,
    description="Specialises in crop-specific watering schedules, micro-irrigation, and water efficiency.",
    instruction="""You are an irrigation engineer advising on crop watering schedules and water efficiency.
Use get_irrigation_advice and get_current_weather tools. If rain is detected, advise skipping irrigation.
First check farmer profile for location/crops. Encourage drip and sprinklers to maximize yield per drop.""",
    tools=[get_irrigation_advice, get_current_weather, get_farmer_profile],
    before_model_callback=safety_guardrail,
)

market_agent = LlmAgent(
    name="market_agent",
    model=MODEL,
    description="Fetches live mandi prices and provides market trend analysis.",
    instruction="""You are an agricultural market analyst. Use get_mandi_price and get_price_trend tools.
State prices as ₹ per quintal. Never guarantee profits. Present data as informational only.
Recommend consulting FPOs for bulk sales. Mention PM-Kisan helpline if farmer is in distress.""",
    tools=[get_mandi_price, get_price_trend, get_farmer_profile],
    before_model_callback=safety_guardrail,
)

government_scheme_agent = LlmAgent(
    name="government_scheme_agent",
    model=MODEL,
    description="Specialises in government agricultural schemes, subsidies, crop insurance, and loans.",
    instruction="""You are a government welfare schemes advisor for Indian farmers.
Use get_scheme_info, list_all_schemes, and check_scheme_eligibility tools.
Guide farmers on step-by-step applications, required documents, official portals, and state/central subsidies.
Check profile for land size and state to evaluate eligibility.""",
    tools=[get_scheme_info, list_all_schemes, check_scheme_eligibility, get_farmer_profile],
    before_model_callback=safety_guardrail,
)

# ── Root Orchestrator ───────────────────────────────────────────────────────

root_agent = LlmAgent(
    name="farming_assistant_router",
    model=MODEL,
    description="Main Krishi Mitra assistant that routes farmer queries to specialist agents.",
    instruction="""You are Krishi Mitra (ಕೃಷಿ ಮಿತ್ರ / कृषि मित्र) — a friendly farming assistant for Indian farmers.
You speak English, Kannada, and Hindi.

FIRST ACTION: Call get_farmer_profile. If empty, ask for location, soil, and crops, then call save_farmer_profile.

LANGUAGE: Kannada → call update_farmer_language('Kannada') | Hindi → call update_farmer_language('Hindi').

ROUTING:
- Weather, rain forecast, temperature → weather_agent
- Irrigation, watering schedule, drip, sprinkler → irrigation_agent
- Soil types, health, NPK, manure, crop choice → soil_agent
- Disease, leaf spot, mildew, blight, photo diagnosis → disease_agent
- Insect pests, caterpillars, aphids, bugs, spraying → pest_agent
- Mandi price, market rates, MSP, APMC → market_agent
- Subsidies, welfare schemes, loans, KCC, PM-KISAN, PMFBY → government_scheme_agent

TONE: Warm, simple, supportive. Emergency → PM-Kisan helpline: 155261""",
    tools=[get_farmer_profile, save_farmer_profile, update_farmer_language],
    sub_agents=[disease_agent, pest_agent, soil_agent, weather_agent, irrigation_agent, market_agent, government_scheme_agent],
    before_model_callback=safety_guardrail,
)

print("✅ All 8 agents built successfully!")

## Step 6: Set Up Runner & Chat Function

In [ ]:
import asyncio
import nest_asyncio
import logging
from google.adk.runners import InMemoryRunner
from google.genai import types

# Apply nest_asyncio to allow asyncio.run() in Jupyter event loop
nest_asyncio.apply()

APP_NAME = "krishi_mitra"
USER_ID = "farmer_demo"
SESSION_ID = "demo_session"

runner = InMemoryRunner(agent=root_agent, app_name=APP_NAME)

async def setup_session():
    await runner.session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)

asyncio.run(setup_session())

async def chat(message: str) -> str:
    """Send a message to Krishi Mitra and get a response."""
    response_text = ""
    message_content = types.Content( 
        role="user",
        parts=[types.Part.from_text(text=message)]
    )
    async for event in runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=message_content):
        if event.is_final_response() and event.content and event.content.parts:
            response_text = event.content.parts[0].text
    return response_text

def ask(message: str):
    """Synchronous wrapper to chat with Krishi Mitra."""
    print(f"\n👨‍🌾 Farmer: {message}")
    print("-" * 60)
    response = asyncio.run(chat(message))
    print(f"🌾 Krishi Mitra: {response}")
    print()
    return response

print("✅ Runner ready! Use ask('your question') to chat with Krishi Mitra.")

---
## 🎯 Summary

This notebook demonstrates a multi-agent farming assistant built with **Google ADK**:

| Agent | Focus Area | Custom Tools Used |
|---|---|---|
| **Disease Agent** | Pathogen diagnosis from symptoms or photographs | `describe_farm_image`, `get_farmer_profile` |
| **Pest Agent** | Insect management and pesticide safety rules | `describe_farm_image`, `get_farmer_profile` |
| **Soil Agent** | Soil testing, NPK fertility, and crop suitability | `recommend_crops`, `get_fertilizer_advice`, `add_crop_to_profile` |
| **Weather Agent** | Weather updates, rain reports, and temperature forecasts | `get_current_weather`, `get_5day_forecast` |
| **Irrigation Agent** | Water needs and schedule adjustments based on rain | `get_irrigation_advice`, `get_current_weather` |
| **Market Agent** | Mandatory live APMC mandi prices and crop selling trends | `get_mandi_price`, `get_price_trend` |
| **Government Scheme Agent** | Central & state welfare benefits, loans, subsidies | `get_scheme_info`, `check_scheme_eligibility` |